<a href="https://colab.research.google.com/github/pop123-ux/DeepSeek-v4-pro-Implementation-using-Gradio/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DeepSeek-v4-pro implementation using Gradio #

In [ ]:
# Install dependencies
!pip install -q gradio openai

In [ ]:
import os
import gradio as gr
from openai import OpenAI
from google.colab import userdata

# Import the api key
try:
  api_key = userdata.get("DEEPSEEK_API_KEY")
except Exception:
  api_key = input('Please enter your DeepSeek API key: ')

client = OpenAI(
    api_key=api_key,
    base_url='https://deepseek.com'
)

# Define the streaming logic
def stream_chat_api(message, history):
  formatted_history = []
  for user_msg, ai_msg in history:
    formatted_history.append({'role': 'user', 'content': user_msg})
    formatted_history.append({'role': 'assistant', 'content': ai_msg})

  formatted_history.append({'role': 'user', 'content': message})

  try:
    response = client.chat.completions.create(
        model='deepseek-v4-pro',
        messages=formatted_history,
        stream=True
    )

    partial_text = ""
    for chunk in response:
      if chunk.choices[0].delta.content:
        partial_text += chunk.choices[0].delta.content
        yield partial_text

  except Exception as e:
    yield f"API Error: {str(e)}"

In [ ]:
# Finally launch gradio with the inline share link
demo = gr.ChatInterface(
    fn=stream_chat_api,
    title='Deepseek API Chatbot inside Colab',
    type='messages'
).launch(share=True) # share=True provides a public URL so you can view the app outside the small Colab cell iframe